# Proposed chronological split counts — 114-case MVP

Use existing metadata and family manifests; no family screening is repeated.
Run all cells with Python 3 from MVP or anywhere inside the repository.
Only the standard library is required.

- Training: **2010–2020**; development: **2021–2022**; final testing: **2023–2024**.
- Check every family member in the **506-source corpus**, including members outside MVP.
- Exclude families spanning period boundaries from the primary comparison.
- Keep recorded calibration families training-only. If a member falls outside the
  training period, exclude the family rather than move dates across periods.
- Hold unresolved companion families, including Sodhi, separately as pending.

Counts use **provisional targets** (0 = affirmed; 1 = reversed/vacated). Human target
and family review remain pending. “Proposed” means passing these date/family rules,
not approval to train or evaluate. This notebook does not freeze or export a split
manifest, assign final splits, change existing data, or train models. The calculation
is deterministic; no random seed is needed for these proposed date-based counts.


In [1]:
from pathlib import Path
from collections import defaultdict, Counter
from datetime import date
import hashlib
import json

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "annotation/source/cases.jsonl").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run inside the legal-judgment repository.")
MVP = ROOT / "annotation/submissions/becca/MVP"
META_PATH = MVP / "data/metadata_v2_family_review/metadata.jsonl"
FAMILY_PATH = MVP / "data/family_v1/case_family_manifest.jsonl"
IDS_PATH = MVP / "data/unanimous_include_v1/case_ids.json"
PROVENANCE_PATH = MVP / "data/family_v1/provenance.json"

def require(ok, message):
    if not ok:
        raise ValueError(message)

def read_rows(p):
    return [json.loads(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]

metadata = read_rows(META_PATH)
family_cases = read_rows(FAMILY_PATH)
selected_ids = json.loads(IDS_PATH.read_text(encoding="utf-8"))
provenance = json.loads(PROVENANCE_PATH.read_text(encoding="utf-8"))
require(hashlib.sha256(META_PATH.read_bytes()).hexdigest() == provenance["metadata_v2_family_review/metadata.jsonl"], "Metadata checksum changed")
require(hashlib.sha256(FAMILY_PATH.read_bytes()).hexdigest() == provenance["outputs"]["case_family_manifest.jsonl"], "Family manifest checksum changed")
require(len(metadata) == len({r["case_id"] for r in metadata}) == len(selected_ids) == len(set(selected_ids)) == 114, "Expected 114 unique MVP IDs")
require({r["case_id"] for r in metadata} == set(selected_ids), "Selected ID mismatch")
require(len(family_cases) == len({r["case_id"] for r in family_cases}) == 506, "Expected 506 source cases")
lookup = {r["case_id"]: r for r in family_cases}
PERIODS = ("train", "development", "test")

def period(value):
    year = date.fromisoformat(value).year
    if 2010 <= year <= 2020: return "train"
    if 2021 <= year <= 2022: return "development"
    if 2023 <= year <= 2024: return "test"
    return "outside_range"

families = defaultdict(list)
for r in family_cases:
    families[r["case_family_id"]].append(r)
print("Verified 114 selected cases and family coverage across 506 source records.")
print("Input SHA-256 checksums:")
for p in (META_PATH, FAMILY_PATH, IDS_PATH, PROVENANCE_PATH):
    print(p.relative_to(MVP), hashlib.sha256(p.read_bytes()).hexdigest())


Verified 114 selected cases and family coverage across 506 source records.
Input SHA-256 checksums:
data/metadata_v2_family_review/metadata.jsonl d4dbb085a0529b5b108f745fe1c53b7e1047ca51e365a541e6f551ee9c1fe641
data/family_v1/case_family_manifest.jsonl 6ef6dadcdcfc27bd63d32430acfade2bc345f95d1be4b3d8124652a35986d103
data/unanimous_include_v1/case_ids.json 16374ff36f54b7b4ece1b219517fe937d6c7e7ffa3fda0b97dac9836d3b47177
data/family_v1/provenance.json cdb88514ca3a8668b5ddf91fd270b484cb9be49de993efa8bd5fe236cfcfe160


## Apply the proposed rules

`case_proposals` retains each case's date period, family, provisional target, all
exclusion reasons, and pending-family flags. Exclusion takes precedence for the
mutually exclusive count categories; pending flags are retained even when excluded.
Labels are used only for reporting counts, never for period assignment.


In [2]:
case_proposals = []
for r in metadata:
    cid = r["case_id"]
    source = lookup[cid]
    require(source["case_family_id"] == r["case_family_id"], f"Family mismatch: {cid}")
    require(source["date_filed"] == r["decision_date"], f"Date mismatch: {cid}")
    require(source["text_sha256"] == r["text_sha256"], f"Source mismatch: {cid}")
    members = families[r["case_family_id"]]
    member_ids = sorted(m["case_id"] for m in members)
    require(member_ids == sorted(source["family_member_case_ids"]), f"Incomplete family: {cid}")
    observed_periods = {period(m["date_filed"]) for m in members}
    training_only = any(m["family_training_only"] for m in members)
    pending_companion = any(m["external_companion_unresolved"] for m in members)
    exclusions = []
    if len(observed_periods) > 1:
        exclusions.append("family_spans_period_boundaries")
    if "outside_range" in observed_periods:
        exclusions.append("family_member_outside_2010_2024")
    if training_only and observed_periods != {"train"}:
        exclusions.append("calibration_family_outside_training_period")
    status = "excluded" if exclusions else "pending" if pending_companion else "proposed"
    require(r["provisional_target"] in (0, 1), f"Missing provisional target: {cid}")
    case_proposals.append({
        "case_id": cid, "case_family_id": r["case_family_id"],
        "decision_date": r["decision_date"], "date_period": period(r["decision_date"]),
        "provisional_target": r["provisional_target"],
        "target_review_status": r["target_review_status"],
        "status": status, "exclusion_reasons": exclusions,
        "pending_reasons": ["external_companion_identity_or_date_unresolved"] if pending_companion else [],
        "training_only": training_only, "family_member_case_ids": member_ids,
        "observed_family_periods": sorted(observed_periods),
    })

proposed = [r for r in case_proposals if r["status"] == "proposed"]
family_periods = defaultdict(set)
for r in proposed:
    family_periods[r["case_family_id"]].add(r["date_period"])
    require(not r["training_only"] or r["date_period"] == "train", "Calibration leakage")
    require(len(r["observed_family_periods"]) == 1, "Cross-period family retained")
require(all(len(v) == 1 for v in family_periods.values()), "Proposed family overlap")
require(len(case_proposals) == 114, "Cases lost during counting")
print("Proposed status counts:", dict(Counter(r["status"] for r in case_proposals)))
print("No known family overlap or calibration leakage in the proposed subset.")


Proposed status counts: {'proposed': 113, 'pending': 1}
No known family overlap or calibration leakage in the proposed subset.


## Counts and sample-size warnings

The table distinguishes all cases dated in each period from the proposed primary
subset. Families and class counts refer to the proposed subset. Excluded and pending
cases remain in the audit. A period is flagged if either class is absent, fewer than
20 cases remain, or a class has fewer than 10 cases. These are transparent screening
heuristics, not a statistical power analysis or a reason to change the date rules.


In [3]:
summary_rows = []
for p in (*PERIODS, "outside_range"):
    all_rows = [r for r in case_proposals if r["date_period"] == p]
    if not all_rows and p == "outside_range": continue
    kept = [r for r in all_rows if r["status"] == "proposed"]
    labels = Counter(r["provisional_target"] for r in kept)
    summary_rows.append({
        "period": p, "all_cases": len(all_rows), "proposed_cases": len(kept),
        "families": len({r["case_family_id"] for r in kept}),
        "class_0": labels[0], "class_1": labels[1],
        "excluded": sum(r["status"] == "excluded" for r in all_rows),
        "pending": sum(r["status"] == "pending" for r in all_rows),
        "training_only": sum(r["training_only"] for r in kept),
    })
columns = ["period", "all_cases", "proposed_cases", "families", "class_0", "class_1", "excluded", "pending", "training_only"]
widths = {c: max(len(c), max(len(str(r[c])) for r in summary_rows)) for c in columns}
print(" | ".join(c.ljust(widths[c]) for c in columns))
print("-+-".join("-" * widths[c] for c in columns))
for r in summary_rows:
    print(" | ".join(str(r[c]).ljust(widths[c]) for c in columns))
require(sum(r["all_cases"] for r in summary_rows) == 114, "Summary total mismatch")
require(all(r["all_cases"] == r["proposed_cases"] + r["excluded"] + r["pending"] for r in summary_rows), "Status counts do not reconcile")
require(all(r["class_0"] + r["class_1"] == r["proposed_cases"] for r in summary_rows), "Class counts do not reconcile")
print("\nSample-size flags (heuristics only):")
for r in summary_rows:
    flags = []
    if min(r["class_0"], r["class_1"]) == 0: flags.append("one or both classes absent")
    if r["proposed_cases"] < 20: flags.append("fewer than 20 cases")
    if min(r["class_0"], r["class_1"]) < 10: flags.append("fewer than 10 cases in at least one class")
    print(r["period"] + ": " + ("; ".join(flags) or "no threshold flag; power not established"))
print("\nExcluded / pending cases:")
for r in case_proposals:
    if r["status"] != "proposed":
        print(r["case_id"], r["date_period"], r["status"], r["exclusion_reasons"] + r["pending_reasons"])
print("\nRecorded training-only cases:", ", ".join(r["case_id"] for r in proposed if r["training_only"]))
print("\nAll counts remain provisional. No training or final split assignment performed.")


period      | all_cases | proposed_cases | families | class_0 | class_1 | excluded | pending | training_only
------------+-----------+----------------+----------+---------+---------+----------+---------+--------------
train       | 82        | 81             | 81       | 55      | 26      | 0        | 1       | 4            
development | 16        | 16             | 16       | 8       | 8       | 0        | 0       | 0            
test        | 16        | 16             | 16       | 11      | 5       | 0        | 0       | 0            

Sample-size flags (heuristics only):
train: no threshold flag; power not established
development: fewer than 20 cases; fewer than 10 cases in at least one class
test: fewer than 20 cases; fewer than 10 cases in at least one class

Excluded / pending cases:
NY-4655053 train pending ['external_companion_identity_or_date_unresolved']

Recorded training-only cases: NY-2757329, NY-4546774, NY-4840329, NY-6004283

All counts remain provisional. No training